# 00. FLIR Termal Veri Seti İndirme ve Hedef Ayrıştırma (ATR Pipeline)

Bu notebook, Hugging Face üzerinde barındırılan Teledyne FLIR termal kamera veri setini indirerek otomatik hedef tanıma (ATR) sınıflandırma mimarisine uygun hale getirir. Orijinal veri setindeki nesne tespiti koordinatları (bounding box) ayrıştırılarak kara aracı (`car`) ve piyade/personel (`person`) sınıflarına ait termal hedefler kırpılır; model eğitim kalitesini düşürmemesi adına 15 pikselden küçük gürültülü alanlar elenir. Elde edilen saf termal görüntüler, PyTorch `ImageFolder` standardına uygun şekilde `dataset/raw/` altında `train` ve `validation` dizinlerine otomatik olarak yapılandırılır.

In [2]:
%pip install datasets tqdm

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/559.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/559.1 kB ? eta -:--:--
   ------------------------------------- -- 524.3/559.1 kB 1.7 MB/s eta 0:00:01
   ---------------------------------------- 559.1/559.1 kB 1.6 MB/s  0:00:00

   ---------------------------------------- 0/3 [dill]
   ---------------------------------------- 0/3 [dill]
   ---------------------------------------- 0/3 [dill]
   ---------------------------------------- 0/3 [dill]
   ------------- -------------------------- 1/3 [multiprocess]
   ------------- -------------------------- 1/3 [multiprocess]
   ------------- -------------------------- 1/3 [multiprocess]
   ------------- -------------------------- 1/3 [multiprocess]
   ------------- -------------------------- 1/3 [multiprocess]
   -------------------------- ------------- 2/3 [datasets]
   -------------------------


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os
from datasets import load_dataset
from tqdm import tqdm

print("FLIR Termal Veri Seti Hugging Face üzerinden indiriliyor...")
dataset = load_dataset("Francesco/flir-camera-objects")

# FLIR hedefleri (2: car, 4: person)
target_classes = {2: "car", 4: "person"}
output_dir = "../dataset/raw"

for split in ["train", "validation"]:
    print(f"\n{split.upper()} verileri kırpılıp klasörleniyor...")
    
    for item in tqdm(dataset[split]):
        img = item["image"]
        objects = item["objects"]
        
        # Görseldeki hedefleri koordinatlarına göre kırp
        for i, (bbox, cat_id) in enumerate(zip(objects["bbox"], objects["category"])):
            if cat_id in target_classes:
                class_name = target_classes[cat_id]
                save_path = os.path.join(output_dir, split, class_name)
                os.makedirs(save_path, exist_ok=True)
                
                # Bbox: [x, y, width, height] -> Crop: [left, upper, right, lower]
                x, y, w, h = bbox
                cropped_img = img.crop((x, y, x + w, y + h))
                
                # Çok küçük kırpmaları filtreleme
                if cropped_img.size[0] > 15 and cropped_img.size[1] > 15:
                    img_id = item["image_id"]
                    cropped_img.save(os.path.join(save_path, f"flir_{img_id}_{i}.jpg"))

print("\nTermal hedefler başarıyla çıkarıldı ve ImageFolder yapısına uygun şekilde klasörlendi.")

C:\Users\sibel\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


FLIR Termal Veri Seti Hugging Face üzerinden indiriliyor...


C:\Users\sibel\AppData\Roaming\Python\Python311\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sibel\.cache\huggingface\hub\datasets--Francesco--flir-camera-objects. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating train split: 9306 examples [00:01, 8215.45 examples/s]
Generating validation split


TRAIN verileri kırpılıp klasörleniyor...


100%|██████████| 9306/9306 [01:09<00:00, 133.04it/s]



VALIDATION verileri kırpılıp klasörleniyor...


100%|██████████| 1452/1452 [00:13<00:00, 111.60it/s]


Termal hedefler başarıyla çıkarıldı ve ImageFolder yapısına uygun şekilde klasörlendi.
